# Calculate Genome-wide Burden for All CNVs, All Genic CNVs, and CNVs Overlapping All Gene-set Genes

In [ ]:
import pandas as pd
import os
import sys
import time
import subprocess

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
%%writefile gene_set_analysis/scripts/calculate_length_burden.R
#!/usr/bin/env Rscript


suppressPackageStartupMessages({
    library(data.table)
})

args <- commandArgs(trailingOnly = TRUE)
if (length(args) != 3) {
    stop("Usage: Rscript calculate_length_burden.R <raw_file_path> <length_map_file> <output_file>")
}

raw_file_path   <- args[1]
length_map_file <- args[2]
output_file     <- args[3]

# 1. LOAD LENGTH MAP
length_map <- fread(length_map_file, header = TRUE)
setnames(length_map, 1:2, c("CNV_ID", "CNV_Length_bp"))
length_map[, CNV_ID := trimws(gsub("_N$", "", CNV_ID))] # Clean early
setkey(length_map, CNV_ID)

# 2. LOAD RAW DATA
# We only need FID, IID, and the marker columns
dt_raw <- fread(raw_file_path, header = TRUE)

# Clean column names immediately to match length_map
original_names <- names(dt_raw)
cleaned_names <- gsub("_N$", "", original_names)
setnames(dt_raw, original_names, cleaned_names)

# Identify marker columns (usually column 7 onwards)
marker_cols <- cleaned_names[7:length(cleaned_names)]

# 3. EFFICIENT RECODING (2 -> 0)
# Instead of lapply (which copies), use set() to modify in-place
for (col in marker_cols) {
    set(dt_raw, i = which(dt_raw[[col]] == 2), j = col, value = 0L)
}

# 4. CALCULATION FUNCTION
# Optimized to use matrix multiplication %*% which is memory-efficient in R
calculate_burden <- function(dt, markers, type_string, len_map) {
    # Filter markers by type (DEL/DUP)
    type_markers <- grep(type_string, markers, value = TRUE, ignore.case = TRUE)
    
    if (length(type_markers) == 0) {
        return(list(length = rep(0, nrow(dt)), count = rep(0, nrow(dt))))
    }
    
    # Align lengths with existing columns
    matched_map <- len_map[type_markers, nomatch = NULL]
    final_markers <- matched_map$CNV_ID
    lengths <- matched_map$CNV_Length_bp
    
    # Convert subset to matrix only once
    # Using %*% (Matrix Multiplication) is much faster than sweep + rowSums
    mat <- as.matrix(dt[, ..final_markers])
    
    # Replace NAs with 0 to prevent propogating NAs in sums
    mat[is.na(mat)] <- 0
    
    list(
        length = (mat %*% lengths) / 1e6,
        count  = rowSums(mat)
    )
}

# 5. PROCESS AND SAVE
final_results <- data.table(FID = dt_raw$FID, IID = dt_raw$IID)

# Deletions
del_res <- calculate_burden(dt_raw, marker_cols, "DEL", length_map)
final_results[, `:=`(total_del_burden_mb = as.vector(del_res$length), 
                     total_del_count = del_res$count)]

# Duplications
dup_res <- calculate_burden(dt_raw, marker_cols, "DUP", length_map)
final_results[, `:=`(total_dup_burden_mb = as.vector(dup_res$length), 
                     total_dup_count = dup_res$count)]

fwrite(final_results, file = output_file, sep = "\t")

# Bash Script to Submit calculate_length_burden.R Script

In [ ]:
%%writefile gene_set_analysis/scripts/cnv_length_burden_batch.sh 
#!/bin/bash
set -euo pipefail

echo "--- Batch CNV burden job started ---"
echo "Input Directory: ${raw_dir}"
echo "Output Directory: ${out_dir}"

# Create a local workspace for outputs
LOCAL_TEMP="./results_temp"
mkdir -p "${LOCAL_TEMP}"

# Loop through ONLY .raw files in the mounted input directory
# This avoids processing the .log files
for raw_path in "${raw_dir}"/*.raw; do
    [ -e "$raw_path" ] || continue # Handle empty directory case
    
    base=$(basename "$raw_path")
    
    # Extract chr name using the pattern '.chr' followed by alphanumeric
    # This works for 'chr1', 'chrX', 'chr22', etc.
    chr_name=$(echo "$base" | grep -oP '(?<=\.chr)[0-9XYM]+')
    
    echo "Processing Chromosome: $chr_name from file: $base"
    
    # Run R script
    # We output to the local temp folder first
    Rscript "${r_script_burden}" \
        "${raw_path}" \
        "${length_map_file}" \
        "${LOCAL_TEMP}/${chr_name}_burden.tsv"
done

# Move all generated TSVs to the dsub output directory
# dsub will then sync these back to your bucket
cp ${LOCAL_TEMP}/*.tsv "${out_dir}/"

echo "--- Batch Job finished ---"

In [ ]:
import subprocess

# --- Configuration ---
R_SCRIPT      = f"{my_bucket}/calculate_length_burden.R"
SHELL_SCRIPT  = f"{my_bucket}/cnv_length_burden_step01.2.sh" 
LENGTH_MAP    = f"{my_bucket}/data/gene_sets/cnv_length_map.tsv"

gs_input_dir = f"{my_bucket}/all_rare_cnvs_to_extract"
OUTPUT_DIR   = f"{my_bucket}/all_rare_cnvs_to_extract"

# --- Exclusion List ---
# These are the chromosomes that SUCCEEDED according to your logs
# success_chrs = {"y", "22", "21", "18", "15"}

# Get list of raw files
raw_files = subprocess.check_output(
    f"gsutil ls {gs_input_dir}/*_all.raw",
    shell=True
).decode().splitlines()

# manual single job submission
# raw_files = ['gs://fc-secure-8f4c9d34-c213-45da-9249-34e08f62688a/data/gene_sets/results/POPMaD_ALL/cnv_counts/all_rare_cnvs_to_extract/AoU_srWGS_SV.v8.chr10_ALL_PoPMaD_rCNV_all.raw']

for raw in raw_files:
    # Extracting chr name (assumes format like .../filename.chr22_all.raw)
    base = raw.split("/")[-1]
    chr_name = base.split(".chr")[1].split("_")[0]
    
    # Skip if already successful
    if chr_name.lower() in success_chrs:
        print(f"Skipping chr{chr_name} (Already Succeeded)")
        continue
    
    out_file = f"{OUTPUT_DIR}/{chr_name}_burden.tsv"
    
    print(f"Submitting job for chr{chr_name}")

    cmd = f"""
    source ~/aou_dsub.bash
    aou_dsub \
      --image rocker/tidyverse:4.4.1 \
      --name BR-{chr_name} \
      --machine-type n2-standard-64 \
      --disk-size 400 \
      --boot-disk-size 100 \
      --input raw_file={raw} \
      --input r_script_burden={R_SCRIPT} \
      --input length_map_file={LENGTH_MAP} \
      --output out_file={out_file} \
      --script "{SHELL_SCRIPT}"
    """

    subprocess.run(cmd, shell=True, executable='/bin/bash')

In [ ]:
import subprocess

# --- Configuration ---
my_bucket     = os.getenv('WORKSPACE_BUCKET')
R_SCRIPT      = f"{my_bucket}/calculate_length_burden.R"
SHELL_SCRIPT  = f"{my_bucket}/cnv_length_burden_batch.sh" 
LENGTH_MAP    = f"{my_bucket}/ndd_cnv_length_map.tsv"

# Parent directory containing the 149 folders (e.g., ndd_10kb_cnv_ids_present_in_set_fatigue)
gs_root_dir   = f"{my_bucket}/results/POPMaD_ALL/cnv_counts"
output_root   = f"{my_bucket}/POPMaD_ALL/burden_scores"

# 1. Get list of all subdirectories (gene sets)
# This lists directories only by looking for the trailing '/'
folders = subprocess.check_output(
    f"gsutil ls -d {gs_root_dir}/*/", 
    shell=True
).decode().splitlines()
# folders = ['gs://fc-secure-8f4c9d34-c213-45da-9249-34e08f62688a/data/gene_sets_ndd/results/POPMaD_ALL/cnv_counts/ndd_10kb_cnv_ids_present_in_set_synaptome/']
# 2. Loop through each gene set folder
for folder_path in folders:
    # Extract folder name (e.g., ndd_10kb_cnv_ids_present_in_set_fatigue)
    folder_name = folder_path.strip("/").split("/")[-1]
    
    # Define specific output folder for this gene set
    current_out_dir = f"{output_root}/{folder_name}"
    
    print(f"Submitting 24-chr batch job for: {folder_name}")
    
    job_name = f"{folder_name.replace('ndd_10kb_cnv_ids_present_in_set_', '')}"

    # dsub command
    # --input-recursive downloads the whole folder to the VM
    # --output-recursive uploads all resulting TSVs back to GCS
    cmd = f"""
    source ~/aou_dsub.bash
    aou_dsub \
      --image rocker/tidyverse:4.4.1 \
      --name {job_name[:30]} \
      --machine-type n2-standard-8 \
      --disk-size 500 \
      --boot-disk-size 100 \
      --input-recursive raw_dir={folder_path} \
      --input r_script_burden={R_SCRIPT} \
      --input length_map_file={LENGTH_MAP} \
      --output-recursive out_dir={current_out_dir} \
      --script "{SHELL_SCRIPT}"
    """

    subprocess.run(cmd, shell=True, executable='/bin/bash')

In [ ]:
# ANCESTRIES = ["POPMaD_ALL" "EUR", "AFR", "AMR"]
ANCESTRIES = ["POPMaD_ALL"] 

R_SCRIPT     = f"{my_bucket}/scripts/merge_and_sum_all_burden3.R"
SHELL_SCRIPT = f"{my_bucket}/cnv_merge_step02.sh" 


for pop in ANCESTRIES:
    INPUT_BASE_DIR      = f"{my_bucket}/data/gene_sets_ndd/results/{pop}"
    BURDEN_SCORES_DIR   = f"{INPUT_BASE_DIR}/burden_scores"
    BURDEN_SUMMARY_DIR  = f"{INPUT_BASE_DIR}/burden_scores_summary"
    LOG_DIR             = f"{BURDEN_SUMMARY_DIR}/logs"

    print(f"\n==== STARTING ANCESTRY: {pop} ====")

    try:
        
        # this part automatically capture folder names and put it inside the loop; however
        # some jobs require higher configs (CPU and RAM) so it is commented
        folder_list_burden_scores = subprocess.check_output(f"gsutil ls {BURDEN_SCORES_DIR}/", shell=True).decode().splitlines()
        GENE_SET_FOLDERS = [f.strip('/').split('/')[-1] for f in folder_list_burden_scores 
                    if f.endswith('/')
                    and ('all' in f or '10kb' in f)]

#         GENE_SET_FOLDERS = ['all_rare_cnvs_to_extract']
        
        print(f"Found {len(GENE_SET_FOLDERS)} gene sets to process for {pop}.")
    except Exception as e:
        print(f"Error finding folders for {pop} (maybe directory doesn't exist?): {e}")
        continue
        
  # --- Loop over detected Gene Sets for this Ancestry ---
    for folder_name in GENE_SET_FOLDERS:
        gs_input_dir = f"{BURDEN_SCORES_DIR}/{folder_name}"
        
        # 1. Create a clean tag
        clean_tag = folder_name.replace('_', '-').replace('.', '-').strip('-')
        final_output_dir = f"{BURDEN_SUMMARY_DIR}/"
        set_identifier = f"{pop}_Burden_Summary_{folder_name}"
        
        # 2. FIX: Truncate job name to 60 chars max
        # We use pop + a shortened version of the folder name
        job_name = f"SM3-{pop}-{clean_tag}"[:60].strip('-')

        print(f"  --> Submitting: {job_name}")

        dsub_cmd = f"""
        source ~/aou_dsub.bash
        aou_dsub \\
          --image rocker/tidyverse:4.4.1 \\
          --name "{job_name}" \\
          --boot-disk-size 50 \\
          --disk-size 150 \\
          --machine-type "n2-standard-4" \\
          --logging "{LOG_DIR}" \\
          --input r_script_merge="{R_SCRIPT}" \\
          --input-recursive gs_input_dir="{gs_input_dir}" \\
          --output-recursive out_dir="{BURDEN_SUMMARY_DIR}" \\
          --env SET_IDENTIFIER="{set_identifier}" \\
          --script "{SHELL_SCRIPT}"
        """

        subprocess.run(dsub_cmd, shell=True, executable="/bin/bash")
        time.sleep(0.5)
        
print(f"\n[DONE] All gene sets for {ANCESTRIES} have been submitted.")

# Aggregate All Sets

In [ ]:
%%writefile gene_set_analysis/scripts/aggregate_all_sets.R
#!/usr/bin/env Rscript
suppressPackageStartupMessages(library(data.table))

args <- commandArgs(trailingOnly = TRUE)
input_dir <- args[1]
output_file <- args[2]

# 1. Identify all the individual result files
all_files <- list.files(path = input_dir, pattern = "POPMaD_ALL_.*\\.tsv$", full.names = TRUE)
message(paste("Found", length(all_files), "files to aggregate."))

if (length(all_files) == 0) {
    stop("No files found! Check your input directory path.")
}

# 2. Read and combine
master_list <- lapply(all_files, function(f) {
    dt <- fread(f)
    # Extract the set name from the filename
    set_name <- gsub("POPMaD_ALL_|.tsv", "", basename(f))
    dt[, gene_set := set_name]
    return(dt)
})

master_dt <- rbindlist(master_list)

# 3. Save the result
fwrite(master_dt, output_file, sep = "\t")
message(paste("Master file saved successfully to:", output_file))

In [ ]:
%%bash

source ~/aou_dsub.bash

# --- Paths ---
R_AGG_SCRIPT="${WORKSPACE_BUCKET}/aggregate_all_sets.R"
INPUT_DIR="${WORKSPACE_BUCKET}/burden_scores_summary/"
FINAL_MASTER_FILE="${WORKSPACE_BUCKET}/results/POPMaD_ALL/burden_scores_summary/TOTAL_COMBINED_NDD_BURDEN_MASTER_all_CNV_sizes.tsv"

echo "--- Submitting Aggregation Job ---"

aou_dsub \
    --image rocker/tidyverse:4.4.1 \
    --name "CNV_AGGREGATE_FINAL" \
    --boot-disk-size 50 \
    --machine-type "n2-standard-16" \
    --input r_script="${R_AGG_SCRIPT}" \
    --input-recursive input_dir="${INPUT_DIR}" \
    --output final_master="${FINAL_MASTER_FILE}" \
    --command 'Rscript "${r_script}" "${input_dir}" "${final_master}"'

echo "Aggregation job submitted. Your final file will be at: ${FINAL_MASTER_FILE}"